In [40]:
import json
import joblib
import pandas as pd
import numpy as np

# Load rule base
with open("D:\\Coding\\6th sem\\Microprocessor\\road_quality\\rules\\hybrid_rst_rules.json", "r") as f:
    rst_rules = json.load(f)

# Load trained ML model
rf_model = joblib.load("D:\\Coding\\6th sem\\Microprocessor\\road_quality\\model\\random_forest_model.pkl")

print("Rules Loaded:", len(rst_rules))

Rules Loaded: 26


In [41]:
import json
import numpy as np

# Load your threshold file
with open("D:\\Coding\\6th sem\\Microprocessor\\road_quality\\rules\\discretization_thresholds.json", "r") as f:
    thresholds = json.load(f)

# Reconstruct full bin edges
bins_dict = {}

for feature, splits in thresholds.items():
    bins_dict[feature] = [-np.inf] + sorted(splits) + [np.inf]

print("Reconstructed Bins:")
for k, v in bins_dict.items():
    print(k, "→", v)

Reconstructed Bins:
acc_std → [-inf, 3051.450439453125, 3947.046142578125, inf]
acc_rms → [-inf, 16569.8271484375, 16918.8330078125, inf]
z_peak_to_peak → [-inf, 17276.0, 19108.0, inf]
speed_mean → [-inf, 6.568700075149536, 18.427949905395508, inf]


In [42]:
#clean
import pandas as pd

LABELS = ["Low", "Medium", "High"]

def discretize_row(row):

    return {
        "acc_std_bin": str(pd.cut(
            [row["acc_std"]],
            bins=bins_dict["acc_std"],
            labels=LABELS
        )[0]),

        "acc_rms_bin": str(pd.cut(
            [row["acc_rms"]],
            bins=bins_dict["acc_rms"],
            labels=LABELS
        )[0]),

        "z_peak_to_peak_bin": str(pd.cut(
            [row["z_peak_to_peak"]],
            bins=bins_dict["z_peak_to_peak"],
            labels=LABELS
        )[0]),

        "speed_mean_bin": str(pd.cut(
            [row["speed_mean"]],
            bins=bins_dict["speed_mean"],
            labels=LABELS
        )[0])
    }

In [43]:
#Rule matching
def match_rule(discretized_row):

    for idx, rule in enumerate(rst_rules):

        if (
            rule["acc_std_bin"] == discretized_row["acc_std_bin"] and
            rule["acc_rms_bin"] == discretized_row["acc_rms_bin"] and
            rule["z_peak_to_peak_bin"] == discretized_row["z_peak_to_peak_bin"] and
            rule["speed_mean_bin"] == discretized_row["speed_mean_bin"]
        ):

            return {
                "rule_id": idx,
                "decision": rule["decision"],
                "support": rule.get("support", "Not Stored")
            }

    return None

In [44]:
#step 4 Hybrid Engine
import numpy as np

def hybrid_predict(row):

    # 1️⃣ Discretize
    discretized = discretize_row(row)

    # 2️⃣ Try Rough Set rule match
    rule_match = match_rule(discretized)

    if rule_match:

        print("\n✅ Deterministic Rule Matched")
        print("Matched Rule ID:", rule_match["rule_id"])
        print("Rule Support:", rule_match["support"])

        return {
            "prediction": rule_match["decision"],
            "method": "RST",
            "confidence": 1.0
        }

    else:
        # 3️⃣ ML fallback
        features = row[[
            "acc_std",
            "acc_rms",
            "z_peak_to_peak",
            "speed_mean"
        ]].values.reshape(1, -1)

        proba = rf_model.predict_proba(features)[0]
        pred = rf_model.predict(features)[0]
        confidence = np.max(proba)

        print("\n⚡ No Rule Match — ML Fallback Used")
        print("ML Confidence:", round(confidence, 4))

        return {
            "prediction": pred,
            "method": "RandomForest",
            "confidence": float(confidence)
        }

In [45]:
df = pd.read_csv("D:\\Coding\\6th sem\\Microprocessor\\road_quality\\preprocessed_data\\road_feature_dataset_50Hz_2sec_50overlap.csv")

result = hybrid_predict(df.iloc[10])

print("\nFinal Prediction:", result)


✅ Deterministic Rule Matched
Matched Rule ID: 3
Rule Support: Not Stored

Final Prediction: {'prediction': 'smooth', 'method': 'RST', 'confidence': 1.0}


In [46]:
#ml fallback
ML_FEATURES = [
    "acc_mean",
    "acc_std",
    "acc_rms",
    "z_max",
    "z_min",
    "z_peak_to_peak",
    "speed_mean",
    "speed_std"
]

In [47]:
# Hybrid Engine updated
import numpy as np
rst_count = 0
ml_count = 0

def hybrid_predict(row):
    global rst_count, ml_count

    discretized = discretize_row(row)
    rule_match = match_rule(discretized)

    if rule_match:
        rst_count += 1
        return rule_match["decision"], "RST"

    else:
        ml_count += 1

        features = pd.DataFrame(
            [row[ML_FEATURES].values],
            columns=ML_FEATURES
        )

        pred = rf_model.predict(features)[0]
        return pred, "RandomForest"

In [48]:
#Run on full dataset
rst_count = 0
ml_count = 0

predictions = []

for i in range(len(df)):
    pred, method = hybrid_predict(df.iloc[i])
    predictions.append(pred)

total = len(df)

print("Total Windows:", total)
print("RST Decisions:", rst_count)
print("ML Fallback:", ml_count)
print("Rule Coverage %:", round((rst_count / total) * 100, 2))

Total Windows: 166
RST Decisions: 134
ML Fallback: 32
Rule Coverage %: 80.72


In [49]:
#compute Hybrid Accuracy
from sklearn.metrics import accuracy_score

hybrid_preds = predictions
true_labels = df["road_label"]

print("Hybrid Accuracy:", accuracy_score(true_labels, hybrid_preds))

Hybrid Accuracy: 0.9819277108433735


In [50]:
#RST deterministic accuracy only 
rst_correct = 0
rst_total = 0

for i in range(len(df)):
    pred, method = hybrid_predict(df.iloc[i])
    if method == "RST":
        rst_total += 1
        if pred == df.iloc[i]["road_label"]:
            rst_correct += 1

print("RST Accuracy:", rst_correct / rst_total)

RST Accuracy: 0.9850746268656716


In [51]:
#ML fall back accuracy only
ml_correct = 0
ml_total = 0

for i in range(len(df)):
    pred, method = hybrid_predict(df.iloc[i])
    if method == "RandomForest":
        ml_total += 1
        if pred == df.iloc[i]["road_label"]:
            ml_correct += 1

print("ML Boundary Accuracy:", ml_correct / ml_total)

ML Boundary Accuracy: 0.96875


In [52]:
import joblib
joblib.dump(rf_model, "D:\\Coding\\6th sem\\Microprocessor\\road_quality\\model\\rf_model_final.pkl")

['D:\\Coding\\6th sem\\Microprocessor\\road_quality\\model\\rf_model_final.pkl']

In [53]:
import json

metrics = {
    "total_windows": 166,
    "rst_decisions": 134,
    "ml_fallback": 32,
    "rule_coverage_percent": 80.72,
    "hybrid_accuracy": 0.9819,
    "rst_accuracy": 0.9851
}

with open("D:\\Coding\\6th sem\\Microprocessor\\road_quality\\outputs\\hybrid_evaluation_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

In [54]:
df["hybrid_prediction"] = predictions
df.to_csv("D:\\Coding\\6th sem\\Microprocessor\\road_quality\\model\\hybrid_results_v1.csv", index=False)

In [55]:
coverage_info = {
    "dependency_theoretical": 0.8133,
    "coverage_empirical": 0.8072
}

with open("D:\\Coding\\6th sem\\Microprocessor\\road_quality\\rules\\coverage_validation.json", "w") as f:
    json.dump(coverage_info, f, indent=4)